In [1]:
import sys 
sys.path.append('C:/Users/DATA/Documents/datos/01_script/inicio/funciones')
from funciones import *
from funciones_spark import *
from variables_inicio import *
from utils_sql import *

credicash

In [2]:
from sqlalchemy import create_engine


engine_mysql = create_engine(
    f"mysql+pymysql://{user_valentina}:{pwd_valentina}@{server_valentina}:{port_mysql}/{db_valentina}"
)

query = """
SELECT 
DISTINCT 
NUMERO_DOCUMENTO as dni_cliente
FROM alfcc_clientes
WHERE cl_base = 'mayo 2026'
and cl_estado=1
"""

df_dni = pd.read_sql(query, engine_mysql)

df_dni["dni_cliente"] = (
    df_dni["dni_cliente"]
    .astype(str)
    .str.zfill(8)
)

In [5]:
exec_query_sql(server_zeus, "MAEBA", user_zeus, pwd_zeus, "ADM_OBJ_TG.spFunnelDinersTc", "SP funnel diners_tc Zeus")

SP funnel diners_tc Zeus | realizado | duración: 19.96 seg


In [ ]:
overwrite_table_SQL(spark,df_prueba_1,f'borrar_TARGET_202604_01',server_kishin,user_kishin,pwd_kishin,'DANTALION')
overwrite_table_SQL(spark,df_prueba_2,f'borrar_TARGET_202604_02',server_kishin,user_kishin,pwd_kishin,'DANTALION')
overwrite_table_SQL(spark,df_prueba_ch,f'borrar_TARGET_202604_ch',server_kishin,user_kishin,pwd_kishin,'DANTALION')


In [7]:
from sqlalchemy import create_engine


engine_mysql = create_engine(
    f"mysql+pymysql://{user_valentina}:{pwd_valentina}@{server_valentina}:{port_mysql}/{db_valentina}"
)

query = """
SELECT 
DISTINCT 
NUMERO_DOCUMENTO as dni_cliente
FROM alfin_clientes
WHERE cl_base = 'abril 2026'
and estado='activo'
"""

df_dni = pd.read_sql(query, engine_mysql)

df_dni["dni_cliente"] = (
    df_dni["dni_cliente"]
    .astype(str)
    .str.zfill(8)
)

In [8]:
filename='RetiroDeGestion_BlackList.csv'

filePath = os.path.join(ruta_csv, filename)

df_list = pd.read_csv(filePath)
df_list.count()


DNI    512682
dtype: int64

In [9]:
print(df_list.columns)
print(df_dni.columns)

Index(['DNI'], dtype='object')
Index(['dni_cliente'], dtype='object')


In [10]:
df_list.rename(columns={'DNI': 'dni_cliente'}, inplace=True)
df_list["dni_cliente"] = (
    df_list["dni_cliente"]
    .astype(str)
    .str.zfill(8)
)
print(df_list.columns)
print(df_dni.columns)

Index(['dni_cliente'], dtype='object')
Index(['dni_cliente'], dtype='object')


In [12]:
df_list = df_list.merge(
    df_dni,
    on="dni_cliente",
    how="inner"
)
df_list['retiro']='Retirar RCC'
df_list.count()


dni_cliente    172
retiro         172
dtype: int64

In [13]:
update_mysql_en_bloques(
    df=df_list,
    tabla="alfin_clientes",
    periodo="abril 2026",
    col_llave_mysql="NUMERO_DOCUMENTO",
    col_valor_mysql="estado",
    col_llave_df="dni_cliente",
    col_valor_df="retiro",
    host=server_valentina,
    user=user_valentina,
    password=pwd_valentina,
    database=db_valentina,
    port=port_mysql,
    batch_size=1000,
    validar_sin_grabar=False
)


Total registros a procesar: 172
Lote 0 - 172 actualizado | filas afectadas: 185
Proceso terminado. Total filas afectadas: 185
